# Projeto Final


## Descrição do projeto

A finalidade do presente projeto é a criação de um modelo de predição do diagnóstico de Hipertiroidismo. A seguir serão apresentadas cada uma das quatro etapas desse desenvolvimento: exploração e preparação dos dados, análise exploratória, seleção e avaliação de modelos e comunicação dos resultado.

Na primeira etapa, exploração e preparação dos dados, estão descritos as correções aplicadas à base de dados. Por vezes, essas correções podem influenciar no resultado final, sendo importante conhecermos o tratamento de dados para compreender o resultado final. A seguir, na etapa de análise exploratória, uma análise visual e estatística dos dados será explorada. Na terceira etapa, serão discutidos as opções de modelo que melhor se adequam aos objetivos do projeto, os modelos serão criados e avaliados. Por fim, o desempenho dos modelos e demais resultados serão apresentados.




## Exploração e Preparação dos dados

trocar dados categoricos f e t por binários
dados faltantes com ?
coluna alvo: binaryclass
P: caso positivo de tiroidismo
N: tiroidismo não identificado

In [472]:
import pandas as pd
import re
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [375]:
data = pd.read_csv('dados.csv')

In [376]:
print(data.head())

  age sex on thyroxine query on thyroxine on antithyroid medication sick  \
0  41   F            f                  f                         f    f   
1  23   F            f                  f                         f    f   
2  46   M            f                  f                         f    f   
3  70   F            t                  f                         f    f   
4  70   F            f                  f                         f    f   

  pregnant thyroid surgery I131 treatment query hypothyroid  ... TT4 measured  \
0        f               f              f                 f  ...            t   
1        f               f              f                 f  ...            t   
2        f               f              f                 f  ...            t   
3        f               f              f                 f  ...            t   
4        f               f              f                 f  ...            t   

   TT4 T4U measured   T4U FTI measured  FTI TBG measured

já de cara vemos que alguns dados estão marcados com "?", que deve ser ausente.
vo investigar mais cuidadosamente a presença de nulos e de "?".
Esses valores não são reconhcidos como nulos, e não nulos na base.
além disso, todos os valores são objetos, como mostram os resultados a seguir

In [377]:
temp = data.isnull().sum()
for i,n in enumerate(temp):
     if n > 0: print(f"{i}: {n}")

In [378]:
tipo_var=['number', 'int64', 'float', 'bool', 'object', 'string']
print(f'Número de variáveis do tipo')
for i in tipo_var:
    print(f'{i}: {data.dtypes[data.dtypes == i].count()}')


Número de variáveis do tipo
number: 0
int64: 0
float: 0
bool: 0
object: 30
string: 0


Os resultados acima mostram que todas as variáveis são objetos, quando sabemos que algumas deveriam ser números. Inicialmente vamos identificar qual deveria ser o tipo de cada variável. Depois todas as variáveis serão transformadas em numéricas, com os devidos métodos aplicados a cada uma. Só depois os valores ausentes serão corrigidos. 

In [379]:
column_number = []
column_letters = []
column_others = []

for i in data.columns:
    amostra = data[i].iloc[0]

    if re.match(r'^\d+$', str(amostra)):
        column_number.append(i)
    
    elif re.match(r'^[a-zA-Z]+$', str(amostra)):
        column_letters.append(i)

    else:
        column_others.append(i)

print("Colunas numéricas:", column_number)
print("Colunas com letras:", column_letters)
print("Outras colunas:", column_others)


Colunas numéricas: ['age', 'TT4', 'FTI']
Colunas com letras: ['sex', 'on thyroxine', 'query on thyroxine', 'on antithyroid medication', 'sick', 'pregnant', 'thyroid surgery', 'I131 treatment', 'query hypothyroid', 'query hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH measured', 'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured', 'TBG measured', 'referral source', 'binaryClass']
Outras colunas: ['TSH', 'T3', 'T4U', 'TBG']


['TSH', 'T3', 'T4U', 'TBG'] foram classificados como outros, então qual será que deveria ser seu valor?


In [380]:
for i in column_others:
    print(f'Valores únicos em {i}: {len(data[i].unique())}')
    print(f'{data[i].head()}')

Valores únicos em TSH: 288
0     1.3
1     4.1
2    0.98
3    0.16
4    0.72
Name: TSH, dtype: object
Valores únicos em T3: 70
0    2.5
1      2
2      ?
3    1.9
4    1.2
Name: T3, dtype: object
Valores únicos em T4U: 147
0    1.14
1       ?
2    0.91
3       ?
4    0.87
Name: T4U, dtype: object
Valores únicos em TBG: 1
0    ?
1    ?
2    ?
3    ?
4    ?
Name: TBG, dtype: object


Pelo resultado, fica claro que 'TSH', 'T3', 'T4U' são numéricos, e que TBG é uma coluna vazia. A coluna TBG será desprezada, pois não acrescenta nenhuma informação, e as três últimas serão anexadas ao grupo de variáveis numéricas. 

In [381]:
column_number.append('TSH')
column_number.append('T3')
column_number.append('T4U')
data.drop('TBG', axis = 1, inplace=True)

transformar todos as variaveis da lista column_number em numéricos. a função tu_nomeric, com o parametro coerce, transformas todos os valores nao numericos, como '?', em nulos.

In [382]:
for i in column_number:
    data[i] = pd.to_numeric(data[i], errors='coerce')
    print(f'\n{i} -> Tipo: {data[i].dtype}, nulos: {data[i].isnull().sum()}, %total: {data[i].isnull().sum()/len(data)*100:.2f}%')


age -> Tipo: float64, nulos: 1, %total: 0.03%

TT4 -> Tipo: float64, nulos: 231, %total: 6.12%

FTI -> Tipo: float64, nulos: 385, %total: 10.21%

TSH -> Tipo: float64, nulos: 369, %total: 9.78%

T3 -> Tipo: float64, nulos: 769, %total: 20.39%

T4U -> Tipo: float64, nulos: 387, %total: 10.26%


COm essa informação em mãos, podemos avaliar o comportamento das variáveis numéricas e escolher qual estratégia será usada para o tratamento dos valores nulos. Após essa etapa, voltaremos etapa de transformação em valores lógicos, numéricos, das variavéis categóricas.

In [394]:
subplotsTitles = []
for i in column_number:
     subplotsTitles.append(f'Histograma de {i}')
     subplotsTitles.append(f'Boxplot de {i}')


fig = make_subplots(rows=len(column_number), cols=2, subplot_titles=subplotsTitles)
row = 1
# col = 1

for i in column_number:
    histogram = px.histogram(data, x=i, nbins=80, histnorm='percent')
    box = px.box(data, y=i)
    fig.add_trace(
        histogram.data[0],
        row=row, col=1,
    )

    fig.add_trace(
        box.data[0],
        row=row, col=2
    )
    row += 1
fig.update_layout(title_text='Histogramas das Colunas Numéricas', showlegend=False, height=3000, width=1300 )
fig.show()



Da análise dos histogramas e dos dados, percebe-se que todos possuem médias muito bem marcadas, com uma frequêncial frequência consideravelmente maior do que as dos de mais valores. Esse comportamento torna seguro a substituição dos valores nulos pela média.
Antes de aplciar a substituição, é preciso tratar também os outliers.

Foram identificados como outlaires
- valor máximo de age
- doi maiores valores de tt4, valores acima de 350
- fti valore acima de 300
- tsh valor acima de 300
- valor t3 acima de 8
- valores negativos de t4u

a riscos quando se exclui valores altos, pois podem ser valores razoáveis que descrevem exatamente a população com índices mais graves e razos.
infelizmente, não tenho certeza de quais são as unidades das medidas, então não é possível comprar com valores conhecidos.

as pesquisas indicaram algumas faixas de valores que poderiam ser usadas de base para determinar quais valores são outliers. mpara algumas variáveis os valores estão muito distantes, então optei por aplicar apenas a análise dos dados sem a comparação com estudos anteriores.

fiz uma mistura, me basiei um pouco nas ref, um pouco na análise do box plot, outro pouco no 75% + 2*std, e no iqr



**TSH: Hormônio estimulante da tireoide (thyroid-stimulating hormone)**
- Hipertiroidismo: abaixo de 0,3 mlU/l; 0 to 0.4 mU/L
- Normal: entre 0,3 e 0,4 mlU/l; 0.4 to 4 mU/L
- Hipotiroidismo: acima de 0,4 mlU/l. 4–10 mU/L	10 mU/L
- 1 semana de vida: valores 15 mUI/L = normal


**T3: Triiodotironina (geralmente T3 total quando não especificado “livre”)**
-  Os valores normais de T3 costumam variar entre 80 e 200 ng/dL,
​

**TT4: T4 total, ou tiroxina total. e T4U (T4 uptake)**
- Teste de captação de T4 pelas proteínas ligadoras, usado para corrigir alterações de proteínas séricas.
- Children up to 5 years old: 0.8 – 2.8 nanograms per deciliter (ng/dL).
- Children 6 to 15 years old: 0.8 – 2.1 ng/dL.
- Male adolescents 16 to 17 years old: 0.8 – 2.8 ng/dL.
- Female adolescents 16 to 17 years old: 0.8 – 1.5 ng/dL.
- Adults over 18 years old: 0.9 – 1.7 ng/dL.
- Free T4 test results are usually accurate. However, certain factors may interfere with the results, including:
    - Certain medications or supplements.
    - Pregnancy.
    - Severe illness or malnourishment.
​

**FTI (Free Thyroxine Index ou Índice de Tiroxina Livre)**
- Índice calculado (T4 total × fator derivado do T4U) para estimar a tiroxina livre antes da ampla disponibilidade do T4 livre direto.
- Em muitos locais, os intervalos considerados normais ficam em torno de:
- 0,7 a 1,8 ng/dL para adultos.

**Referências**
- https://my.clevelandclinic.org/health/diagnostics/24235-thyroxine-t4-test
- https://www.rededorsaoluiz.com.br/exames-e-procedimentos/analises-clinicas/indice-tiroxina-livre
- https://www.saudebemestar.pt/pt/exame/analises-clinicas/tsh/
- https://sergiofranco.com.br/saude/exame-de-tireoide-tsh
- https://fleminglaboratorio.com/glossario/exames-tireoide-valores-normais/

In [329]:
# Resumo estatístico dos dados
print(data[column_number].describe())

               age          TT4          FTI          TSH           T3  \
count  3771.000000  3541.000000  3387.000000  3403.000000  3003.000000   
mean     51.735879   108.319345   110.469649     5.086766     2.013500   
std      20.084958    35.604248    33.089698    24.521470     0.827434   
min       1.000000     2.000000     2.000000     0.005000     0.050000   
25%      36.000000    88.000000    93.000000     0.500000     1.600000   
50%      54.000000   103.000000   107.000000     1.400000     2.000000   
75%      67.000000   124.000000   124.000000     2.700000     2.400000   
max     455.000000   430.000000   395.000000   530.000000    10.600000   

               T4U  
count  3385.000000  
mean      0.995000  
std       0.195457  
min       0.250000  
25%       0.880000  
50%       0.980000  
75%       1.080000  
max       2.320000  


In [384]:
diqr = pd.DataFrame(columns=['min', 'max', 'upper_bound'])
for i in column_number:
    q1 = data[i].quantile(0.25)
    q3 = data[i].quantile(0.75)
    iqr = q3 - q1
    # lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    diqr.loc[i] = [data[i].min(), data[i].max(), upper_bound]

print(diqr)

       min     max  upper_bound
age  1.000  455.00       113.50
TT4  2.000  430.00       178.00
FTI  2.000  395.00       170.50
TSH  0.005  530.00         6.00
T3   0.050   10.60         3.60
T4U  0.250    2.32         1.38


voltanto, os valores que serão EXCLUÍDOS são:
Foram identificados como outlaires
- valor máximo de age
- doi maiores valores de tt4, valores acima de 350
- fti valore acima de 300
- tsh valor acima de 300
- valor t3 acima de 8
- nenhum valor em t4u

In [385]:
data = data[data['age'] < 100]
data = data[data['TT4'] < 200]
data = data[data['FTI'] < 200]
data = data[data['TSH'] < 30]
data = data[data['T3'] < 8]
data = data[data['T4U'] < 8]
print(data.shape)

(2597, 29)


In [386]:
for i in column_number:
    data[i] = pd.to_numeric(data[i], errors='coerce')
    print(f'\n{i} -> Nulos: {data[i].isnull().sum()}, %total: {data[i].isnull().sum()/len(data)*100:.2f}%')


age -> Nulos: 0, %total: 0.00%

TT4 -> Nulos: 0, %total: 0.00%

FTI -> Nulos: 0, %total: 0.00%

TSH -> Nulos: 0, %total: 0.00%

T3 -> Nulos: 0, %total: 0.00%

T4U -> Nulos: 0, %total: 0.00%


outra estratégia para o tratamento de dados poderia ser excluir os campos que não tem todas as medidas, em que  TSH measured, T3 measured, TT4 measured: , TT4 measured, FTI measured e TBG measured  são falsos. A estratégia utilizado buscou exercitar a análise numérica e comportamental das variáveis. Mas a outra estratégia seria mais ágil .

Feito então, substitui nulos dos numéricos, corrigi outlieres, agora é a vez de olharmos novamente para as variaveis categóricas

verificou-se que não há erros de digitação nem variações de escrita para os mesmos valores.
a classe sex possui '?', será subsituído por "NI", de não informado, not informed.
a classe sex referral source será transformada com uma adaptação da técnica  Label Encoder, em que Atribui um número único a cada valor único na sua coluna categórica, cata categoria vira um número inteiro,  memória, pois usa apenas uma coluna para representar a variável.
as de mais com o valor f será transformado em 0 e o valor t em 1., p =1 n =0,

 m=1, f =0, ?=2

as categorias TSH measured, T3 measured, TT4 measured: , TT4 measured, FTI measured e TBG measured serão eliminadas por que contém apenas um valor, t, que indica que a medida foi feita.

In [387]:
for i in column_letters:
    print(f'{i}: {data[i].unique()}')

sex: ['F' 'M' '?']
on thyroxine: ['f' 't']
query on thyroxine: ['f' 't']
on antithyroid medication: ['f' 't']
sick: ['f' 't']
pregnant: ['f' 't']
thyroid surgery: ['f' 't']
I131 treatment: ['f' 't']
query hypothyroid: ['f' 't']
query hyperthyroid: ['f' 't']
lithium: ['f' 't']
goitre: ['f' 't']
tumor: ['f' 't']
hypopituitary: ['f' 't']
psych: ['f' 't']
TSH measured: ['t']
T3 measured: ['t']
TT4 measured: ['t']
T4U measured: ['t']
FTI measured: ['t']
TBG measured: ['f']
referral source: ['SVHC' 'SVI' 'other' 'STMW' 'SVHD']
binaryClass: ['P' 'N']


In [430]:
data = data.drop(['TSH measured', 'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured', 'TBG measured'], axis = 1)
column_letters.remove('TSH measured') 
column_letters.remove('T3 measured')
column_letters.remove('TT4 measured')
column_letters.remove('T4U measured')
column_letters.remove('FTI measured')
column_letters.remove('TBG measured')

KeyError: "['TSH measured', 'T3 measured', 'TT4 measured', 'FTI measured', 'TBG measured'] not found in axis"

In [389]:
for i in column_letters:
        data[i] = data[i].replace('t', 1)
        data[i] = data[i].replace('f', 0)

        data[i] = data[i].replace('M', 1)
        data[i] = data[i].replace('F', 0)
        data[i] = data[i].replace('?', 2)

        data[i] = data[i].replace('P', 1)
        data[i] = data[i].replace('N', 0)

        data[i] = data[i].replace('other', 0)
        data[i] = data[i].replace('STMW', 1)
        data[i] = data[i].replace('SVHC', 2)
        data[i] = data[i].replace('SVHD', 3)
        data[i] = data[i].replace('SVI', 4)

for i in column_letters:
    print(f'{i}: {data[i].unique()}')

sex: [0 1 2]
on thyroxine: [0 1]
query on thyroxine: [0 1]
on antithyroid medication: [0 1]
sick: [0 1]
pregnant: [0 1]
thyroid surgery: [0 1]
I131 treatment: [0 1]
query hypothyroid: [0 1]
query hyperthyroid: [0 1]
lithium: [0 1]
goitre: [0 1]
tumor: [0 1]
hypopituitary: [0 1]
psych: [0 1]
referral source: [2 4 0 1 3]
binaryClass: [1 0]


C:\Users\marina.freitas\AppData\Local\Temp\ipykernel_8172\1792849895.py:7: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\marina.freitas\AppData\Local\Temp\ipykernel_8172\1792849895.py:3: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\marina.freitas\AppData\Local\Temp\ipykernel_8172\1792849895.py:16: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior,

pronto,com as variáveis categóricas numéricas, agora podemos avaliar comportamentos e balanceamento das variáveis


In [395]:
fig = make_subplots(rows=6, cols=3, subplot_titles=column_letters)

row = 1
col = 1

for i in column_letters:
    histogram = px.histogram(data, x=i, nbins=10, histnorm='percent')
    fig.add_trace(
        histogram.data[0],
        row=row, col=col
    )
    col += 1
    if col > 3:
        col = 1
        row += 1
fig.update_layout(title_text='Histogramas das Colunas de Letras', showlegend=False,     height=2000, width=1300 )
fig.show()

p resultado já indica que as variáveis de treino terão que ser balanceadas.
então esse será o próximo passo, separar em treino e teste e balancear.
alvo = hyperthyroidism


from sklearn.preprocessing import StandardScaler



from sklearn.model_selection import cross_val_score

normalizar
penguins_padrao = penguins.copy()
colunas_padronizar = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
scaler = StandardScaler()
penguins_padrao[colunas_padronizar] = scaler.fit_transform(penguins_padrao[colunas_padronizar])




In [436]:
x = data.drop('query hyperthyroid', axis = 1)
y = data['query hyperthyroid']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [466]:
px.histogram(y_train, x='query hyperthyroid', nbins=3, histnorm='percent', title='Distribuição da variável alvo no conjunto de treino', width=1000).show()


veja como a variável alvo de treino está desbalanceada. a solução será aplciar o smote
SMOTE (Synthetic Minority Over-sampling Technique)
 Técnica de oversampling que cria exemplos sintéticos da classe minoritária
 Baseado na interpolação entre exemplos existentes da classe minoritária

In [468]:
smote = SMOTE(random_state=42)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train, y_train)
train_balance = y_train_balanced.value_counts()
print(f'Balanceamento em y_train: {train_balance}')

Balanceamento em y_train: query hyperthyroid
0    1717
1    1717
Name: count, dtype: int64


Avaliando os valores máximos das diferentes categorias explanatorias, percebem-se valores muito discrepentes. para evitar que as diferenças entre seus valores absolutos interfiram no resultado, vamos padronizar com a ténica de standard scaler

In [471]:
x_train.describe()

,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,...,tumor,hypopituitary,psych,TSH,T3,TT4,T4U,FTI,referral source,binaryClass
count,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,...,1817.000000,1817.0,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000,1817.000000
mean,52.711613,0.391304,0.094662,0.009906,0.013209,0.038525,0.015960,0.012658,0.019813,0.056136,...,0.018162,0.0,0.068244,2.347986,1.982389,106.683544,0.992741,108.379196,1.758943,0.936709
std,18.960963,0.552718,0.292827,0.099064,0.114198,0.192513,0.125357,0.111825,0.139395,0.230248,...,0.133573,0.0,0.252234,3.659699,0.699280,27.648050,0.185607,24.700386,1.795494,0.243553
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.005000,0.050000,3.000000,0.360000,3.000000,0.000000,0.000000
25%,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.500000,1.600000,89.000000,0.880000,93.000000,0.000000,1.000000
50%,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,1.300000,2.000000,103.000000,0.970000,106.000000,2.000000,1.000000
75%,69.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,2.500000,2.300000,122.000000,1.080000,121.000000,4.000000,1.000000
max,94.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,0.0,1.000000,29.000000,7.100000,199.000000,1.830000,198.000000,4.000000,1.000000


In [473]:
scaler = StandardScaler()
x_train_balanced_scaled = scaler.fit_transform(x_train_balanced)
x_test_scaled = scaler.transform(x_test)

movida por boas práticas, vou salvar essa versao dos arquivos 

In [475]:
y_train_balanced.to_csv('y_train_balanced.csv', index=False)
x_train_balanced.to_csv('x_train_balanced.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
x_test.to_csv('x_test.csv', index=False)

# Parei aqui: selecionar o modelo de machine learning
## Seleção e avaliação de modelos

problema de classificação multiclasse 
minimizar os falsos negativos para garantir que nenhum caso passe desbercebido
análise visual

Falsos positivos e negativos

Termos usados para descrever erros em modelos de classificação. Falsos positivos ocorrem quando o modelo prevê incorretamente a presença de uma condição, enquanto falsos negativos ocorrem quando o modelo não detecta uma condição que está presente.

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import ConfusionMatrixDisplay


## Comunicação dos resultado